# Webinar 2: Data Preprocessing — Track 1: Tabular Pipeline
### Dataset: Real IBM Telco Customer Churn (7,043 customer records)
### Algorithms: XGBoost & LightGBM vs Baseline Logistic Regression

This notebook covers the complete tabular preprocessing lifecycle on real-world business data:
1. **Profiling**: Missing value audit in `TotalCharges`, extreme outliers in `MonthlyCharges`, skewness, and cardinality.
2. **Encoding**: One-Hot Encoding for nominal columns (`PaymentMethod`, `InternetService`), Ordinal Encoding for `Contract`.
3. **Scaling**: Comparing `StandardScaler`, `MinMaxScaler`, and `RobustScaler` on skewed billing features.
4. **Imbalance**: Handling the ~27% churn minority class using **SMOTE**.
5. **Train Model & Evaluation**: Comparing naive baseline Logistic Regression vs preprocessed **XGBoost Classifier**.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.tabular import (
    profile_dataframe,
    print_profiling_report,
    TabularEncoder,
    TabularScaler,
    compare_scalers,
    balance_dataset,
    run_tabular_pipeline
)

pd.set_option('display.max_columns', None)
print('Imports loaded successfully!')

Imports loaded successfully!


## Step 1: Data Profiling & Health Audit (IBM Telco Churn)
Inspecting missing values, string anomalies, distributions, and class imbalance.

In [2]:
df_raw = pd.read_csv('../data/tabular/telco_churn_raw.csv')
print(f'IBM Telco Dataset Shape: {df_raw.shape}')
df_raw.head()

IBM Telco Dataset Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Convert TotalCharges to numeric for profiling
df_profile = df_raw.copy()
df_profile['TotalCharges'] = pd.to_numeric(df_profile['TotalCharges'].astype(str).str.strip(), errors='coerce')
profile = profile_dataframe(df_profile, target_col='Churn')
print_profiling_report(profile)

TABULAR DATA HEALTH & PROFILING AUDIT REPORT
Total Records: 7043 | Total Features: 21

--- 1. Missing Values & Feature Types ---
          column   dtype  null_count  null_pct  unique_values type_category
      customerID     str           0      0.00           7043   categorical
          gender     str           0      0.00              2   categorical
   SeniorCitizen   int64           0      0.00              2       numeric
         Partner     str           0      0.00              2   categorical
      Dependents     str           0      0.00              2   categorical
          tenure   int64           0      0.00             73       numeric
    PhoneService     str           0      0.00              2   categorical
   MultipleLines     str           0      0.00              3   categorical
 InternetService     str           0      0.00              3   categorical
  OnlineSecurity     str           0      0.00              3   categorical
    OnlineBackup     str           

## Step 2: Categorical Encoding (Nominal vs Ordinal)
- Nominal features (`PaymentMethod`, `InternetService`, etc.) $\to$ **One-Hot Encoding**
- Ordinal feature (`Contract`: Month-to-month < One year < Two year) $\to$ **Ordinal Encoding**

In [4]:
# Prepare binary and categorical features
X_clean = df_raw.drop(columns=['Churn', 'customerID'])
X_clean['TotalCharges'] = pd.to_numeric(X_clean['TotalCharges'].astype(str).str.strip(), errors='coerce')
y = df_raw['Churn'].map({'Yes': 1, 'No': 0})

nominal_cols = ['PaymentMethod', 'InternetService', 'OnlineSecurity', 'DeviceProtection']
ordinal_cols = ['Contract']
ordinal_order = {'Contract': ['Month-to-month', 'One year', 'Two year']}

encoder = TabularEncoder(
    nominal_cols=nominal_cols,
    ordinal_cols=ordinal_cols,
    ordinal_categories=ordinal_order
)
X_encoded = encoder.fit_transform(X_clean)
print(f'Encoded Feature Matrix Shape: {X_encoded.shape}')
X_encoded.head()

Encoded Feature Matrix Shape: (7043, 14)


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Contract_ordinal,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,DeviceProtection_No internet service,DeviceProtection_Yes
0,0,1,29.85,29.85,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0,34,56.95,1889.50,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0,2,53.85,108.15,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0,45,42.30,1840.75,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0,2,70.70,151.65,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Step 3: Feature Scaling Comparison (Standard vs MinMax vs Robust)
Comparing scaler resistance to extreme charges.

In [5]:
scaler_comparison = compare_scalers(X_encoded, num_cols=['MonthlyCharges', 'tenure'])
scaler_comparison

,Scaler,Feature,Mean,Std,Min,Median,Max,IQR
0,Raw (No Scaling),MonthlyCharges,64.7617,30.0879,18.2500,70.3500,118.7500,54.3500
1,Raw (No Scaling),tenure,32.3711,24.5577,0.0000,29.0000,72.0000,46.0000
2,StandardScaler (Z-Score),MonthlyCharges,-0.0000,1.0000,-1.5459,0.1857,1.7944,1.8064
3,StandardScaler (Z-Score),tenure,-0.0000,1.0000,-1.3182,-0.1373,1.6137,1.8731
4,"MinMaxScaler ([0, 1])",MonthlyCharges,0.4628,0.2994,0.0000,0.5184,1.0000,0.5408
5,"MinMaxScaler ([0, 1])",tenure,0.4496,0.3411,0.0000,0.4028,1.0000,0.6389
6,RobustScaler (IQR-based),MonthlyCharges,-0.1028,0.5536,-0.9586,0.0000,0.8905,1.0000
7,RobustScaler (IQR-based),tenure,0.0733,0.5339,-0.6304,0.0000,0.9348,1.0000


In [6]:
from src.evaluation.visualizer import plot_tabular_scaling_and_outliers
plot_tabular_scaling_and_outliers(df_profile, num_col='MonthlyCharges', output_path='../reports/tabular_scaling_and_outliers.png')

scaler = TabularScaler(method='robust')
X_scaled = scaler.fit_transform(X_encoded)
X_scaled.head()

-> Saved scaling comparison plot to: ../reports/tabular_scaling_and_outliers.png


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Contract_ordinal,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,InternetService_Fiber optic,InternetService_No,OnlineSecurity_No internet service,OnlineSecurity_Yes,DeviceProtection_No internet service,DeviceProtection_Yes
0,0.0,-0.608696,-0.745170,-0.404100,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.108696,-0.246550,0.145381,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0.0,-0.586957,-0.303588,-0.380964,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.347826,-0.516099,0.130977,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
4,0.0,-0.586957,0.006440,-0.368111,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


## Step 4: Class Imbalance Handling with SMOTE

In [7]:
print('Original Churn Distribution:')
print(y.value_counts(normalize=True) * 100)

X_balanced, y_balanced = balance_dataset(X_scaled, y, method='smote')
print('\nBalanced Distribution After SMOTE:')
print(y_balanced.value_counts())

Original Churn Distribution:
Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64

Balanced Distribution After SMOTE:
Churn
0    5174
1    5174
Name: count, dtype: int64


## Step 5: Model Training (XGBoost vs Baseline) & Evaluation

In [8]:
tabular_results = run_tabular_pipeline('../data/tabular/telco_churn_raw.csv')

from src.evaluation.comparison import generate_modality_comparison
comp_df = generate_modality_comparison(tabular_results['baseline_metrics'], tabular_results['preprocessed_metrics'], 'Tabular (Telco Churn)')
comp_df

  Tabular | Baseline: Logistic Regression (Baseline) -> Preprocessed: XGBoost Classifier


,Modality,Metric,Baseline (Before),Preprocessed (After),Delta,Relative Improvement
0,Tabular (Telco Churn),Accuracy,0.8041,0.7825,-0.0216,-2.7%
1,Tabular (Telco Churn),Precision,0.6640,0.5857,-0.0783,-11.8%
2,Tabular (Telco Churn),Recall,0.5289,0.6146,+0.0857,+16.2%
3,Tabular (Telco Churn),F1-Score (Macro),0.7301,0.7252,-0.0049,-0.7%
4,Tabular (Telco Churn),F1-Score (Minority),0.5888,0.5998,+0.0110,+1.9%
5,Tabular (Telco Churn),ROC-AUC,0.8378,0.8370,-0.0008,-0.1%
